In [1]:
!git lfs install


Git LFS initialized.


In [2]:
!git clone https://huggingface.co/Qwen/Qwen2.5-0.5B


Cloning into 'Qwen2.5-0.5B'...


In [3]:
!ls -lh Qwen2.5-0.5Bs


'ls' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
import os
p = "Qwen2.5-0.5B"
for f in sorted(os.listdir(p)):
    fp = os.path.join(p, f)
    if os.path.isfile(fp):
        print(f"{f:<30} {os.path.getsize(fp)/1e6:>10.2f} MB")

.gitattributes                       0.00 MB
LICENSE                              0.01 MB
README.md                            0.00 MB
config.json                          0.00 MB
generation_config.json               0.00 MB
merges.txt                           1.82 MB
model.safetensors                  988.10 MB
tokenizer.json                       7.33 MB
tokenizer_config.json                0.01 MB
vocab.json                           2.78 MB


In [5]:
%cd Qwen2.5-0.5B
!git lfs pull
%cd ..

c:\Users\Rasulbekk\Desktop\LLM_Post_Training\Qwen2.5-0.5B
c:\Users\Rasulbekk\Desktop\LLM_Post_Training


In [7]:
!pip install transformers

  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.6 MB 743.3 kB/s eta 0:00:17
   - -------------------------------------- 0.5/12.6 MB 743.3 kB/s eta 0:00:17
   -- ------------------------------------- 0.8/12.6 MB 771.6 kB/s eta 0:00:16
   --- ------------------------------------ 1.0/12.6 MB 792.7 kB/s eta 0:00:15
   --- ------------------------------------ 1.0/12.6 MB 792.7 kB/s eta 0:00:15
   ---- ----------------------------------- 1.3/12.6 MB 793.9 kB/s eta 0:00:15
   ---- ----------------------------------- 1.3/12.6 MB 793.9 kB/s eta 0:00:15
   ---- ----------------------------------- 1.3/12.6 MB 793.9 kB/s eta 0:00:15
   ---- ----

In [8]:
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

cuda = torch.cuda.is_available()
dtype = torch.float16 if cuda else torch.float32   # CPU'da float16 ishlamaydi

tok = AutoTokenizer.from_pretrained("Qwen2.5-0.5B")
model = AutoModelForCausalLM.from_pretrained("Qwen2.5-0.5B", torch_dtype=dtype).eval()
if cuda:
    model.cuda()

tok.pad_token = tok.pad_token or tok.eos_token
print("Qurilma:", model.device)

c:\Users\Rasulbekk\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 215.00it/s]

Qurilma: cpu


In [9]:
SAVOLLAR = [
    "Python nima?",
    "O‘zbekiston poytaxti qayer?",
    "Menga samarali o‘qish uchun 3 ta maslahat ber.",
    "Bugun havo issiq — shu gapni ingliz tiliga tarjima qil.",
    "2 + 2 * 3 nechaga teng?",
]

def javob(savol):
    inp = tok(savol, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=100, do_sample=False,
                         repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp.input_ids.shape[-1]:], skip_special_tokens=True).strip()

natijalar = [{"savol": s, "base_javob": javob(s)} for s in SAVOLLAR]

for r in natijalar:
    print("SAVOL:", r["savol"])
    print("JAVOB:", r["base_javob"])
    print("-" * 70)

json.dump(natijalar, open("baseline_1dars.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("Saqlandi: baseline_1dars.json")

SAVOL: Python nima?
JAVOB: - YouTube\n0:00 / 12:59
----------------------------------------------------------------------
SAVOL: O‘zbekiston poytaxti qayer?
JAVOB: (O’zbekiston: O’zbekistān qaytət qeriyi) is a 2013 Uzbek film directed by Nadeem Khojaev. It was released on 26 September 2013 in Uzbekistan and has an international release date of 25 October 2013.
Can we infer the following?
The film was made in Uzbekistan.

Yes, sweetie! We can say that the film "
----------------------------------------------------------------------
SAVOL: Menga samarali o‘qish uchun 3 ta maslahat ber.
JAVOB: O’qishlar, qo’shishlar, kashgarmalar. Qo’shishlar, kashgarmalar, qo’shishlar. O’qishlar, qo’shishlar, kashgarmalar. O’qishlar, qo’shishlar, kashgarmalar. O’qishlar, qo’shishlar, kashgarmalar. O’qish
----------------------------------------------------------------------
SAVOL: Bugun havo issiq — shu gapni ingliz tiliga tarjima qil.
JAVOB: Tashkil, kuni 100000000000000000000000000000000000000000000000

In [10]:
natijalar = []
for i, s in enumerate(SAVOLLAR, 1):
    print(f"[{i}/5]...")
    natijalar.append({"savol": s, "base_javob": javob(s)})

[1/5]...
[2/5]...
[3/5]...
[4/5]...
[5/5]...
